# Marching Cubes Surface Extraction
This example demonstrates the complete end-to-end pipeline for converting implicit Signed Distance Fields (SDF) back into explicit mesh geometry.

## Isosurface Generation
We generate a dense Cartesian grid, compute the SDF utilizing a baseline sphere, compute grid normals using GPU central difference, and deploy Conquer3D's lightning-fast GPU Marching Cubes implementation to extract a precise zero-isosurface triangle mesh. Finally, the extracted geometry is visualized against the original.

In [1]:
import torch
import conquer3d as c3d
import plotly.graph_objects as go
import time
from plotly.subplots import make_subplots

# 1. Construct a sphere mesh natively
print("Constructing Sphere...")
verts, tris = c3d.creation.create_sphere(sectors=32, stacks=16, radius=1.0)
mesh = c3d.data_structure.TriangleMesh(verts.cuda(), tris.cuda())

Constructing Sphere...


In [2]:
# 2. Construct voxel grid
print("Constructing Voxel Grid...")
res = 12  # Lower resolution so visualization of normals is actually legible
grid_vertices, voxels, idx_grids = c3d.data_structure.create_voxel_grid(
    grid_min=[-1.2, -1.2, -1.2], 
    grid_max=[1.2, 1.2, 1.2], 
    res=[res, res, res], 
    device="cuda"
)

# 3. Obtain SDF values
print("Computing SDF...")
torch.cuda.synchronize()
t0 = time.time()
_, _, _, sdf_values = mesh.query_points(grid_vertices, return_sdf=True, sign_mode=0)
torch.cuda.synchronize()
print(f"SDF computed in {time.time()-t0:.4f}s")

Constructing Voxel Grid...
Computing SDF...
SDF computed in 0.0021s


In [3]:
# 4. Compute Grid Normals & Visualize Grid
grid_normals = c3d.data_structure.compute_grid_normal(sdf_values, grid_vertices, idx_grids, res=[res, res, res])

# We visualize the grid vertices and normals
gv_np = grid_vertices.cpu().numpy()
gn_np = grid_normals.cpu().numpy()
vox_np = voxels.cpu().numpy()

# Draw voxel wireframe
edge_x, edge_y, edge_z = [], [], []
for v in vox_np:
    # A voxel has 8 corners. We draw the 12 edges.
    corners = gv_np[v]
    # bottom face
    for i in range(4):
        p1, p2 = corners[i], corners[(i+1)%4]
        edge_x.extend([p1[0], p2[0], None])
        edge_y.extend([p1[1], p2[1], None])
        edge_z.extend([p1[2], p2[2], None])
    # top face
    for i in range(4):
        p1, p2 = corners[i+4], corners[(i+1)%4 + 4]
        edge_x.extend([p1[0], p2[0], None])
        edge_y.extend([p1[1], p2[1], None])
        edge_z.extend([p1[2], p2[2], None])
    # vertical edges
    for i in range(4):
        p1, p2 = corners[i], corners[i+4]
        edge_x.extend([p1[0], p2[0], None])
        edge_y.extend([p1[1], p2[1], None])
        edge_z.extend([p1[2], p2[2], None])

fig_grid = go.Figure()
fig_grid.add_trace(go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    mode='lines',
    line=dict(color='gray', width=1),
    name='Grid Wireframe',
    opacity=0.3
))

# Lines for normal vectors
n_edge_x, n_edge_y, n_edge_z = [], [], []
scale = 0.1
for i in range(len(gv_np)):
    n_edge_x.extend([gv_np[i, 0], gv_np[i, 0] + gn_np[i, 0] * scale, None])
    n_edge_y.extend([gv_np[i, 1], gv_np[i, 1] + gn_np[i, 1] * scale, None])
    n_edge_z.extend([gv_np[i, 2], gv_np[i, 2] + gn_np[i, 2] * scale, None])

fig_grid.add_trace(go.Scatter3d(
    x=n_edge_x, y=n_edge_y, z=n_edge_z,
    mode='lines',
    line=dict(color='royalblue', width=2),
    name='Normal Lines'
))

# Cones for normal vectors tips
fig_grid.add_trace(go.Cone(
    x=gv_np[:, 0] + gn_np[:, 0] * scale, 
    y=gv_np[:, 1] + gn_np[:, 1] * scale, 
    z=gv_np[:, 2] + gn_np[:, 2] * scale,
    u=gn_np[:, 0], v=gn_np[:, 1], w=gn_np[:, 2],
    sizemode="absolute",
    sizeref=0.03,
    anchor="tip",
    colorscale="Blues",
    showscale=False,
    name="Normal Cones"
))

fig_grid.update_layout(title="Voxel Grid and Grid Normals", scene=dict(aspectmode='data', camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))))
fig_grid.show()

In [4]:
# 5. Perform Marching Cubes
print("Extracting surface with Marching Cubes...")
torch.cuda.synchronize()
t0 = time.time()
# We reuse the grid_vertices, voxels, sdf_values, and grid_normals computed earlier
grid_colors = (grid_vertices + 1.2) / 2.4 # Map [-1.2, 1.2] to [0, 1] RGB colors
mc_verts, mc_tris, mc_normals, mc_colors = c3d.ops.marching_cubes(grid_vertices, voxels, sdf_values, grid_normals, grid_colors=grid_colors, iso=0.0)
torch.cuda.synchronize()
print(f"Extracted {mc_verts.shape[0]} vertices and {mc_tris.shape[0]} triangles in {time.time()-t0:.4f}s")

Extracting surface with Marching Cubes...
Extracted 376 vertices and 748 triangles in 0.0014s


In [5]:
recon_mesh = c3d.data_structure.TriangleMesh(mc_verts, mc_tris)
print("Is Edge Manifold:", recon_mesh.is_edge_manifold())
print("Is Vertex Manifold:", recon_mesh.is_vertex_manifold())
print("Is Self Intersecting:", recon_mesh.is_self_intersection())
print("Is Fully Manifold:", recon_mesh.is_manifold())
print("Euler Characteristic:", recon_mesh.euler_characteristic)
print("Genus:", recon_mesh.genus)

Is Edge Manifold: True
Is Vertex Manifold: True
Is Self Intersecting: False
Is Fully Manifold: True
Euler Characteristic: 2
Genus: 0


In [6]:
# 6. Visualize Extracted Mesh side-by-side
fig = make_subplots(
    rows=1, cols=3,
    specs=[[{'type': 'scene'}, {'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=('Original Sphere', 'Marching Cubes Extracted (with Normals)', 'Colored Mesh')
)

verts_np = verts.cpu().numpy()
tris_np = tris.cpu().numpy()
mc_verts_np = mc_verts.cpu().numpy()
mc_tris_np = mc_tris.cpu().numpy()
mc_normals_np = mc_normals.cpu().numpy()
mc_colors_np = mc_colors.cpu().numpy()

# Draw Original Sphere Wireframe
orig_x, orig_y, orig_z = [], [], []
for tri in tris_np:
    v0, v1, v2 = verts_np[tri[0]], verts_np[tri[1]], verts_np[tri[2]]
    orig_x.extend([v0[0], v1[0], v2[0], v0[0], None])
    orig_y.extend([v0[1], v1[1], v2[1], v0[1], None])
    orig_z.extend([v0[2], v1[2], v2[2], v0[2], None])

fig.add_trace(go.Scatter3d(
    x=orig_x, y=orig_y, z=orig_z,
    mode='lines',
    line=dict(color='gray', width=2),
    name='Base Wireframe'
), row=1, col=1)

# Draw Marching Cubes Wireframe
mc_x, mc_y, mc_z = [], [], []
for tri in mc_tris_np:
    v0, v1, v2 = mc_verts_np[tri[0]], mc_verts_np[tri[1]], mc_verts_np[tri[2]]
    mc_x.extend([v0[0], v1[0], v2[0], v0[0], None])
    mc_y.extend([v0[1], v1[1], v2[1], v0[1], None])
    mc_z.extend([v0[2], v1[2], v2[2], v0[2], None])

fig.add_trace(go.Scatter3d(
    x=mc_x, y=mc_y, z=mc_z,
    mode='lines',
    line=dict(color='teal', width=2),
    name='MC Wireframe'
), row=1, col=2)

# Lines for Extracted Mesh Normals
n_edge_x, n_edge_y, n_edge_z = [], [], []
scale = 0.2
for i in range(len(mc_verts_np)):
    n_edge_x.extend([mc_verts_np[i, 0], mc_verts_np[i, 0] + mc_normals_np[i, 0] * scale, None])
    n_edge_y.extend([mc_verts_np[i, 1], mc_verts_np[i, 1] + mc_normals_np[i, 1] * scale, None])
    n_edge_z.extend([mc_verts_np[i, 2], mc_verts_np[i, 2] + mc_normals_np[i, 2] * scale, None])

fig.add_trace(go.Scatter3d(
    x=n_edge_x, y=n_edge_y, z=n_edge_z,
    mode='lines',
    line=dict(color='darkred', width=2),
    name='Normal Lines'
), row=1, col=2)

# Cones for Extracted Mesh Normals
fig.add_trace(go.Cone(
    x=mc_verts_np[:, 0] + mc_normals_np[:, 0] * scale, 
    y=mc_verts_np[:, 1] + mc_normals_np[:, 1] * scale, 
    z=mc_verts_np[:, 2] + mc_normals_np[:, 2] * scale,
    u=mc_normals_np[:, 0], v=mc_normals_np[:, 1], w=mc_normals_np[:, 2],
    sizemode="absolute",
    sizeref=0.05,
    anchor="tip",
    colorscale="Reds",
    showscale=False,
    name="Normal Cones"
), row=1, col=2)



# Draw Colored Marching Cubes Mesh
fig.add_trace(go.Mesh3d(
    x=mc_verts_np[:, 0], y=mc_verts_np[:, 1], z=mc_verts_np[:, 2],
    i=mc_tris_np[:, 0], j=mc_tris_np[:, 1], k=mc_tris_np[:, 2],
    vertexcolor=(mc_colors_np * 255).astype('uint8'),
    opacity=0.8,
    name='MC Colored Mesh'
), row=1, col=3)

fig.update_layout(
    scene=dict(aspectmode='data', camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))),
    scene2=dict(aspectmode='data', camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))),
    scene3=dict(aspectmode='data', camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))),
    title="Conquer3D Sphere vs. Extracted Marching Cubes"
)
fig.show()

## Reconstruction Error
Finally, we can evaluate the quality of our Marching Cubes extraction by sampling point clouds from both the ground truth sphere mesh and the extracted surface mesh, and then measuring the discrepancy using our ultra-fast GPU Chamfer Distance implementation!

In [7]:
# 7. Evaluate Reconstruction using Chamfer Distance
print("\n--- Evaluating Extracted Mesh Quality ---")
# Create TriangleMesh for the extracted mesh
mc_mesh = c3d.data_structure.TriangleMesh(mc_verts.cuda(), mc_tris.cuda())

# Sample point clouds
N_samples = 100000
print(f"Sampling {N_samples} points from both ground truth and extracted meshes...")
torch.cuda.synchronize()
t_samp0 = time.time()
gt_pts, *_ = mesh.sample_points(N_samples, uniform=True)
mc_pts, *_ = mc_mesh.sample_points(N_samples, uniform=True)
torch.cuda.synchronize()
print(f"Sampling completed in {time.time()-t_samp0:.4f}s")

# Compute Chamfer Distance
torch.cuda.synchronize()
t_chamfer0 = time.time()
chamfer_dist = c3d.ops.chamfer_distance(gt_pts, mc_pts, squared=False)
torch.cuda.synchronize()

print(f"Reconstruction Chamfer Distance (un-squared L2): {chamfer_dist.item():.8f}")
print(f"Distance computed in {time.time()-t_chamfer0:.5f}s")


--- Evaluating Extracted Mesh Quality ---
Sampling 100000 points from both ground truth and extracted meshes...
Sampling completed in 0.0066s
Reconstruction Chamfer Distance (un-squared L2): 0.02342580
Distance computed in 0.01707s
